# Projeto IANÃ: Inferência LLM - Arquitetura Híbrida v2

**Objetivo:** Executar a extração de dados clínicos de prontuários não-estruturados (MIMIC-IV) utilizando Inteligência Artificial (LLM) guiada por Tipagem Forte (Strict Typing).

### A Arquitetura v2
Para garantir a máxima qualidade dos dados extraídos, esta arquitetura realiza a fusão de duas frentes do projeto:
1. **Estrutura de Leitura (O Padrão SOAP Expandido):** Divide o prontuário em Subjetivo, Objetivo (Exame Físico + Laboratório + Imagem), Avaliação e Plano - permitindo extração detalhada de dados objetivos.
2. **Rigor Científico (A Taxonomia SemClinBR):** Utiliza as categorias clínicas oficiais aprovadas pela equipe de curadoria (ex: `Diagnostic Procedure`, `Laboratory or Test Result`, `Organism or Virus`).

### Melhorias v2 (baseadas nos feedbacks de avaliação):
- **SOAP Objetivo subdividido** em 3 campos (exame físico, laboratório, imagem) para captura completa de dados objetivos
- **Prompt reescrito** removendo a instrução "conciso" que limitava a extração, substituída por instruções exaustivas
- **Field descriptions enriquecidas** com exemplos e instruções explícitas de "TODAS as entidades"
- **Regras explícitas** para negações e doenças previamente tratadas

Com o uso da biblioteca `instructor`, a LLM é forçada a retornar os dados estritamente neste formato, garantindo que o banco de dados receba informações ricas, categorizadas e prontas para pesquisa.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# ==========================================
# 1. O NER (Categorias Oficiais do SemClinBR)
# ==========================================
class EntidadeClinica(BaseModel):
    disease_or_syndrome: List[str] = Field(
        default_factory=list,
        description="TODAS as doenças, diagnósticos e condições clínicas mencionadas no texto, incluindo comorbidades e diagnósticos secundários (ex: Anemia, Hipertensão, Pericardite). Não omita nenhuma."
    )
    sign_or_symptom: List[str] = Field(
        default_factory=list,
        description="TODOS os sinais e sintomas relatados ou observados no exame físico. Inclua achados positivos e exclua os negados pelo paciente (ex: se 'nega febre', NÃO inclua febre)."
    )
    pharmacologic_substance: List[str] = Field(
        default_factory=list,
        description="TODOS os medicamentos citados no prontuário: admissão, curso hospitalar e alta. Inclua nome e dosagem quando disponível (ex: Atenolol 50 mg, Prednisona 40 mg)."
    )
    laboratory_or_test_result: List[str] = Field(
        default_factory=list,
        description="TODOS os resultados laboratoriais com nome do exame E valor numérico. Inclua cada exame individualmente (ex: 'WBC-11.5', 'CD4-113', 'Ferritina-1324', 'D-Dimer-3813'). Não resuma nem agrupe."
    )
    diagnostic_procedure: List[str] = Field(
        default_factory=list,
        description="TODOS os procedimentos diagnósticos e terapêuticos realizados: exames de imagem, biópsias, cirurgias, lavagens, cateterismos (ex: CXR, TTE, CTA, lavagem broncoalveolar, amputação abaixo do joelho)."
    )
    organism_or_virus: List[str] = Field(
        default_factory=list,
        description="TODOS os vírus, bactérias, fungos ou parasitas citados no texto, independentemente do resultado ser positivo ou negativo (ex: HIV, Pneumocystis jirovecii, Mycobacterium tuberculosis)."
    )

# ==========================================
# 2. O S.O.A.P. (Estrutura do Documento)
# ==========================================
class EstruturaSOAP(BaseModel):
    subjetivo: str = Field(
        description="História da doença atual, queixa principal, relato do paciente, histórico médico relevante e revisão de sistemas. Inclua o que o paciente relata E o que nega."
    )
    objetivo_exame_fisico: str = Field(
        description="Achados detalhados do exame físico: sinais vitais completos (T, PA, FC, FR, SpO2), inspeção, ausculta, palpação. Transcreva todos os achados relevantes."
    )
    objetivo_laboratorio: str = Field(
        description="TODOS os resultados laboratoriais com valores numéricos: hemograma, bioquímica, sorologias, culturas, gasometria, marcadores. Liste cada exame individualmente, não resuma."
    )
    objetivo_imagem: str = Field(
        description="TODOS os achados de exames de imagem: raio-X, tomografia, ecocardiograma, etc. Inclua a impressão/conclusão de cada exame."
    )
    avaliacao: str = Field(
        description="Raciocínio clínico completo: diagnóstico principal, diagnósticos diferenciais considerados, evolução do quadro durante a internação e conclusões da equipe médica."
    )
    plano: str = Field(
        description="Conduta completa: medicações prescritas na alta (com dose e posologia), orientações ao paciente, encaminhamentos, exames de seguimento e retornos programados."
    )

# ==========================================
# 3. CONTRATO FINAL
# ==========================================
class RelatorioClinicoProcessado(BaseModel):
    paciente_id: str = Field(description="ID do paciente ou da internação (hadm_id).")
    codigo_cid: str = Field(description="Código CID (ICD) associado.")
    doenca_alvo_identificada: str = Field(description="A doença principal (TB, HIV ou Sífilis).")
    ner: EntidadeClinica
    soap: EstruturaSOAP

print("✅ Arquitetura Híbrida v2 carregada com sucesso!")

### ⚙️ Configuração Segura e Execução (Prova de Conceito)

**Segurança da API:** Para evitar o vazamento de credenciais no repositório, este script utiliza a biblioteca `python-dotenv`. 
Para rodar localmente, certifique-se de criar um arquivo `.env` na raiz do projeto contendo a variável: `OPENAI_API_KEY=sk-sua-chave-aqui`.

Abaixo, simulamos o comportamento de Batch Inference (Processamento em Lote) utilizando 3 amostras isoladas (`.txt`) para validar o pipeline fim a fim.

In [ ]:
import instructor
from openai import OpenAI
import json
import time
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURAÇÃO DO CLIENTE LLM (Via .env)
# ==========================================
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv() 

# Puxa a chave sem expor no código
CHAVE_API_PROFESSOR = os.getenv("OPENAI_API_KEY")

if not CHAVE_API_PROFESSOR:
    raise ValueError("❌ ERRO: Chave API não encontrada. Verifique seu arquivo .env!")

client = instructor.from_openai(
    OpenAI(api_key=CHAVE_API_PROFESSOR),
    mode=instructor.Mode.JSON
)

# ==========================================
# 2. PROMPT DO SISTEMA (Instruções para a LLM)
# ==========================================
SYSTEM_PROMPT = """Você é um especialista clínico de alto nível com experiência em doenças infecciosas (HIV, Tuberculose, Sífilis).
Leia o prontuário clínico em inglês e extraia as informações estritamente no formato JSON exigido.

REGRAS DE EXTRAÇÃO:
1. TRADUÇÃO: Traduza o resumo do SOAP e todas as entidades extraídas para o português brasileiro.
2. NER EXAUSTIVO - Não há limite de entidades. Extraia TODAS as que encontrar no texto:
   - Liste CADA exame laboratorial individualmente com seu valor numérico (ex: WBC-5.0, Hgb-13.9, Creat-1.0).
   - Liste TODOS os medicamentos, incluindo os da admissão, do curso hospitalar e da alta.
   - Liste TODAS as doenças e condições, incluindo comorbidades e diagnósticos secundários (ex: Anemia, DRGE).
   - Liste TODOS os procedimentos realizados (cirurgias, biópsias, exames de imagem).
   - Liste TODOS os organismos testados, mesmo que o resultado seja negativo.
3. NEGAÇÕES: Se o paciente NEGA um sintoma (ex: "denies fever"), NÃO inclua esse sintoma na lista de sinais/sintomas.
4. DOENÇAS PRÉVIAS: Se uma doença foi tratada no passado e não está ativa (ex: "gonorrhea, successfully treated"), NÃO inclua como doença atual.
5. SOAP DETALHADO:
   - Subjetivo: Inclua queixa principal, HDA completa e revisão de sistemas (positivos E negativos).
   - Objetivo (Exame Físico): Transcreva TODOS os achados do exame físico com sinais vitais completos.
   - Objetivo (Laboratório): Liste CADA resultado laboratorial individualmente com valores. Não resuma.
   - Objetivo (Imagem): Inclua a impressão/conclusão de CADA exame de imagem realizado.
   - Avaliação: Inclua raciocínio clínico, diagnósticos diferenciais e evolução.
   - Plano: Inclua medicações com dose/posologia, encaminhamentos e seguimento.
6. PRECISÃO: Não invente dados que não estejam no texto. É melhor extrair uma entidade a mais do que perder uma."""

# ==========================================
# 3. A LISTA DE ARQUIVOS (O "Lote" / Batch)
# ==========================================
# Aqui colocamos os 3 arquivos que você tem. 
# Na DGX, isso será substituído pelas 700 linhas do .parquet
arquivos_para_processar = [
    'amostras/amostra_HIV_CID_042_subj_18170517_hadm_21473169.txt',
    'amostras/amostra_Sifilis_CID_0940_subj_18263451_hadm_27306123.txt',
    'amostras/amostra_Tuberculose_CID_A1884_subj_18734137_hadm_22413268.txt'
]

# Vamos criar uma lista vazia para guardar os 3 JSONs no final
banco_de_dados_final = []

print("🚀 INICIANDO O PROCESSAMENTO EM LOTE (BATCH INFERENCE)...\n")

# ==========================================
# 4. O LOOP MÁGICO (O que a DGX vai fazer)
# ==========================================
for caminho_arquivo in arquivos_para_processar:
    print(f"⏳ Lendo e processando: {caminho_arquivo}")
    
    try:
        # Abre o arquivo da vez
        with open(caminho_arquivo, 'r', encoding='utf-8') as f:
            texto_prontuario = f.read()

        # Manda para a IA
        resultado_ia = client.chat.completions.create(
            model="gpt-4o-mini",
            response_model=RelatorioClinicoProcessado,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": texto_prontuario
                }
            ],
            temperature=0.1
        )
        
        # Converte a resposta validada pelo Pydantic para um dicionário Python e guarda na nossa lista
        banco_de_dados_final.append(resultado_ia.model_dump())
        
        print(f"✅ Sucesso! Paciente processado.\n")
        
        # Pausa de 2 segundos para não sobrecarregar a API (Boa prática!)
        time.sleep(2)

    except FileNotFoundError:
        print(f"❌ ERRO: O arquivo '{caminho_arquivo}' não foi encontrado na pasta. Pulando para o próximo...\n")
    except Exception as e:
        print(f"❌ ERRO ao processar a IA: {e}\n")

# ==========================================
# 5. SALVANDO O RESULTADO FINAL
# ==========================================
print("======================================================")
print("🎉 PROCESSAMENTO CONCLUÍDO! 🎉")
print(f"Total de prontuários processados com sucesso: {len(banco_de_dados_final)}")
print("======================================================")

# Vamos salvar tudo em um único arquivo JSON, simulando o banco de dados
nome_arquivo_saida = 'resultados/banco_dados_iana_amostras.json'
with open(nome_arquivo_saida, 'w', encoding='utf-8') as f:
    json.dump(banco_de_dados_final, f, indent=2, ensure_ascii=False)

print(f"💾 O resultado dos 3 pacientes foi salvo no arquivo: {nome_arquivo_saida}")

### 🚀 Script de Produção (Pipeline DGX)

Esta célula contém o script oficial de **Batch Inference** para ser executado no servidor DGX. 
Ele lê o arquivo `.parquet` contendo o lote completo de prontuários filtrados (Tuberculose, HIV, Sífilis) e itera sobre cada linha, enviando o texto para o motor LLM.

**Instruções para rodar na DGX:**
1. Certifique-se de que o arquivo `mimic_filtrado_tb_hiv_sifilis.parquet` está no mesmo diretório.
2. Certifique-se de que a variável de ambiente `OPENAI_API_KEY` está configurada no arquivo `.env`.
3. Caso a DGX utilize um modelo hospedado localmente (ex: Gemma, Llama 3), altere o `model` e adicione o parâmetro `base_url` no cliente da OpenAI.

In [ ]:
import polars as pl
import instructor
from openai import OpenAI
import json
import time
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURAÇÃO SEGURA (Via .env)
# ==========================================
load_dotenv() 
CHAVE_API_PROFESSOR = os.getenv("OPENAI_API_KEY")

if not CHAVE_API_PROFESSOR:
    raise ValueError("❌ ERRO: Chave API não encontrada no arquivo .env!")

client = instructor.from_openai(
    OpenAI(api_key=CHAVE_API_PROFESSOR),
    mode=instructor.Mode.JSON
)

# ==========================================
# 2. PROMPT DO SISTEMA (Mesmo da POC)
# ==========================================
SYSTEM_PROMPT_DGX = """Você é um especialista clínico de alto nível com experiência em doenças infecciosas (HIV, Tuberculose, Sífilis).
Leia o prontuário clínico em inglês e extraia as informações estritamente no formato JSON exigido.

REGRAS DE EXTRAÇÃO:
1. TRADUÇÃO: Traduza o resumo do SOAP e todas as entidades extraídas para o português brasileiro.
2. NER EXAUSTIVO - Não há limite de entidades. Extraia TODAS as que encontrar no texto:
   - Liste CADA exame laboratorial individualmente com seu valor numérico (ex: WBC-5.0, Hgb-13.9, Creat-1.0).
   - Liste TODOS os medicamentos, incluindo os da admissão, do curso hospitalar e da alta.
   - Liste TODAS as doenças e condições, incluindo comorbidades e diagnósticos secundários (ex: Anemia, DRGE).
   - Liste TODOS os procedimentos realizados (cirurgias, biópsias, exames de imagem).
   - Liste TODOS os organismos testados, mesmo que o resultado seja negativo.
3. NEGAÇÕES: Se o paciente NEGA um sintoma (ex: "denies fever"), NÃO inclua esse sintoma na lista de sinais/sintomas.
4. DOENÇAS PRÉVIAS: Se uma doença foi tratada no passado e não está ativa (ex: "gonorrhea, successfully treated"), NÃO inclua como doença atual.
5. SOAP DETALHADO:
   - Subjetivo: Inclua queixa principal, HDA completa e revisão de sistemas (positivos E negativos).
   - Objetivo (Exame Físico): Transcreva TODOS os achados do exame físico com sinais vitais completos.
   - Objetivo (Laboratório): Liste CADA resultado laboratorial individualmente com valores. Não resuma.
   - Objetivo (Imagem): Inclua a impressão/conclusão de CADA exame de imagem realizado.
   - Avaliação: Inclua raciocínio clínico, diagnósticos diferenciais e evolução.
   - Plano: Inclua medicações com dose/posologia, encaminhamentos e seguimento.
6. PRECISÃO: Não invente dados que não estejam no texto. É melhor extrair uma entidade a mais do que perder uma.
7. O ID de internação (hadm_id) deste paciente é: {hadm_id}. Certifique-se de usá-lo no campo paciente_id."""

# ==========================================
# 3. CARREGANDO O BANCO DE DADOS (Parquet)
# ==========================================
caminho_parquet = 'dados/mimic_filtrado_tb_hiv_sifilis.parquet'
banco_de_dados_final_dgx = []

print("🚀 INICIANDO O PROCESSAMENTO EM LOTE NA DGX...")

try:
    df_mimic = pl.read_parquet(caminho_parquet)
    total_pacientes = df_mimic.height
    print(f"📊 Total de pacientes carregados do Parquet: {total_pacientes}\n")
    
    # ==========================================
    # 4. O LOOP DE PRODUÇÃO (Batch Processing)
    # ==========================================
    for index, row in enumerate(df_mimic.iter_rows(named=True)):
        # Extraindo os dados da linha atual
        hadm_id = row.get('hadm_id', 'Desconhecido')
        texto_prontuario = row.get('text', '')
        
        print(f"⏳ Processando paciente {index + 1}/{total_pacientes} (ID: {hadm_id})...")
        
        try:
            # Enviamos o texto para a IA, injetando o ID no prompt
            resultado_ia = client.chat.completions.create(
                model="gpt-4o-mini", # Alterar aqui se a DGX for usar outro modelo local
                response_model=RelatorioClinicoProcessado,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT_DGX.format(hadm_id=hadm_id)
                    },
                    {
                        "role": "user",
                        "content": texto_prontuario
                    }
                ],
                temperature=0.1
            )
            
            # Converte e salva
            banco_de_dados_final_dgx.append(resultado_ia.model_dump())
            print(f"✅ Sucesso! Dados estruturados.\n")
            
            # Pausa para não estourar os limites da API (Rate Limit)
            time.sleep(1) 

        except Exception as e:
            print(f"❌ ERRO na API ao processar paciente {hadm_id}: {e}\n")

    # ==========================================
    # 5. SALVANDO O RESULTADO OFICIAL
    # ==========================================
    nome_arquivo_saida_dgx = 'resultados/banco_dados_iana_oficial.json'
    with open(nome_arquivo_saida_dgx, 'w', encoding='utf-8') as f:
        json.dump(banco_de_dados_final_dgx, f, indent=2, ensure_ascii=False)

    print("======================================================")
    print("🎉 PROCESSAMENTO DGX CONCLUÍDO COM SUCESSO! 🎉")
    print(f"💾 O banco de dados final ({len(banco_de_dados_final_dgx)} pacientes) foi salvo em: {nome_arquivo_saida_dgx}")
    print("======================================================")

except FileNotFoundError:
    print(f"❌ ERRO FATAL: Arquivo '{caminho_parquet}' não encontrado no diretório. O pipeline foi interrompido.")